In [1]:
from pathlib import Path

import numpy as np
from ls_mcmc import logging, sampling

from cardiac_electrophysiology import mcmc_builder, posterior_builder
from cardiac_electrophysiology.ls_bip import laplace

In [2]:
posterior_settings = posterior_builder.PosteriorBuilderSettings(
    paths=posterior_builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_mcmc_logfile.log"),
    ),
    prior_parameters=posterior_builder.PriorParameters(
        kappa=0.05,
        tau=10,
        seed=0,
    ),
    eikonal_parameters=posterior_builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1,
    ),
    observation_parameters=posterior_builder.ObservationParameters(
        num_observations=1000,
        noise_variance=5e-3,
        seed=0,
    ),
    logger_settings=posterior_builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)
builder = posterior_builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = builder.build(return_additional_data=True)

In [3]:
map_estimate = np.load("../results/map_estimate.npy")
lowrank_data = np.load("../results/lowrank_data.npz")
laplace_approximation = laplace.LaplaceApproximation(
    map_estimate=map_estimate,
    hessian_eigenvalues=lowrank_data["hessian_eigenvalues"],
    hessian_eigenvectors=lowrank_data["hessian_eigenvectors"],
    prior_covariance_eigenvalues=lowrank_data["prior_covariance_eigenvalues"],
    prior_covariance_eigenvectors=lowrank_data["prior_covariance_eigenvectors"],
    prior=posterior.prior,
)
builder_settings = mcmc_builder.MCMCBuilderSettings(
    mcmc_model_settings=mcmc_builder.MCMCModelSettings(
        log_posterior=posterior,
        laplace_approximation=laplace_approximation,
        reference_point=map_estimate,
        step_width=1e-3,
        index_to_track=42,
    ),
    logging_settings=logging.LoggerSettings(
        do_printing=True,
        logfile_path=Path("../results/lsmcmc_logfile.log"),
    ),
    storage_path=Path("../results/mcmc_samples.zarr"),
    storage_chunk_size=10,
    overwrite_existing_storage=True,
)
builder = mcmc_builder.MCMCBuilder(builder_settings)
mcmc_sampler = builder.build()

In [4]:
sampler_settings=sampling.SamplerRunSettings(
    num_samples=1000,
    initial_state=map_estimate,
    print_interval=1,
    checkpoint_path=None,
)
storage, outputs = mcmc_sampler.run(sampler_settings)

| Iteration   | Time        | Accept Rate    | Component 42   | Run_mean_C_42  | 
--------------------------------------------------------------------------------
| 0.000e+00   | 1.651e-03   | +1.000e+00     | -9.389e-02     | -9.389e-02     | 
| 1.000e+00   | 1.785e+00   | +1.000e+00     | -9.392e-02     | -9.390e-02     | 
| 2.000e+00   | 2.629e+00   | +6.667e-01     | -9.392e-02     | -9.391e-02     | 
| 3.000e+00   | 3.471e+00   | +5.000e-01     | -9.392e-02     | -9.391e-02     | 
| 4.000e+00   | 4.317e+00   | +6.000e-01     | -9.354e-02     | -9.384e-02     | 
| 5.000e+00   | 5.155e+00   | +6.667e-01     | -9.324e-02     | -9.374e-02     | 
| 6.000e+00   | 6.002e+00   | +7.143e-01     | -9.399e-02     | -9.378e-02     | 
| 7.000e+00   | 6.882e+00   | +6.250e-01     | -9.399e-02     | -9.380e-02     | 
| 8.000e+00   | 7.719e+00   | +6.667e-01     | -9.358e-02     | -9.378e-02     | 
| 9.000e+00   | 8.580e+00   | +6.000e-01     | -9.358e-02     | -9.376e-02     | 
| 1.000e+01   | 9